## Q1: How many launches of each type?
The first letter of LaunchCode is GCAT's launch category, so I aggregated against that to see the full mix. I want to focus on Orbital Launches.

In [0]:
%sql
SELECT LEFT(TRIM(LaunchCode), 1) AS category,
       COUNT(*) AS n
FROM launch_reliability.bronze.launch
GROUP BY 1
ORDER BY n DESC;

7196 Orbital Launches! I am thinking of keeping deep space launches as well. Curious what ! even is.

## Q2: Which outcome codes appear among orbital launches and deep space launches?
Time to dig a little deeper! Characters that suffix the cateogry indicate the launch outcome. OS is Success and OF is failure. Anything with a number is a partial outcome, OE is a pad explosion, OA is pad abort.

In [0]:
%sql
SELECT TRIM(LaunchCode) AS launch_code,
       COUNT(*) AS n
FROM launch_reliability.bronze.launch
WHERE TRIM(LaunchCode) LIKE 'O%'
GROUP BY 1
ORDER BY n DESC;

In [0]:
%sql
SELECT TRIM(LaunchCode) AS launch_code,
       COUNT(*) AS n
FROM launch_reliability.bronze.launch
WHERE TRIM(LaunchCode) LIKE 'D%'
GROUP BY 1
ORDER BY n DESC;

Okay this is a bit suspicious to me. I am sure there have been failures. For example:
Date	| Mission	| What happened
| -------- | -------- | -------- |
1958 Aug 17	| Pioneer 0 (Moon) |	Rocket exploded about a minute after launch
1960 Oct 10	| Soviet Mars probe	| Rocket failed during ascent
1960 Oct 14	| Soviet Mars probe	| Rocket failed during ascent
1962 Jul 22	| Mariner 1 (Venus)	| Rocket veered off course and was destroyed

In [0]:
%sql
SELECT Launch_Tag, Launch_Date, LV_Type, Flight, Mission, TRIM(LaunchCode) AS launch_code
FROM launch_reliability.bronze.launch
WHERE Launch_Date LIKE '1958 Aug 17%'
   OR Launch_Date LIKE '1960 Oct 10%'
   OR Launch_Date LIKE '1960 Oct 14%'
   OR Launch_Date LIKE '1962 Jul 22%';

Interesting! It seems that all 4 of those failures are classified as OF. I will combine O and D launches into one population. If a D launch fails, it appears to just be classified as an orbital failure.

## Q3: What do the dates look like?
Since I will likely be using time in my analysis (comparing old space and new space), I will take a look at the dates!

In [0]:
%sql
SELECT Launch_Date, Launch_JD
FROM launch_reliability.bronze.launch
WHERE TRIM(LaunchCode) LIKE 'O%'
ORDER BY rand()
LIMIT 15;

In [0]:
%sql
SELECT COUNT(*) AS missing_jd
FROM launch_reliability.bronze.launch
WHERE LEFT(TRIM(LaunchCode), 1) IN ('O', 'D')
  AND (TRIM(Launch_JD) IN ('-', '') OR Launch_JD IS NULL);

It appears the launch_date is in varying formats, but the Julian Date looks clean and not missing for any of the rows. I will convert that to a normal timestamp with `unix_seconds = (Launch_JD - 2440587.5) × 86400`

# Silver Code

In [0]:
#helper to clean 
from pyspark.sql import functions as F

def clean_col(name):
    """Trim padding; turn '-' and '' into null."""
    c = F.trim(F.col(name))
    return F.when(c.isin("-", ""), None).otherwise(c)

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("launch_reliability.bronze.launch")

launch = (bronze
    # 1. clean and select
    .select(
        clean_col("Launch_Tag").alias("launch_tag"),
        F.trim("Launch_JD").cast("double").alias("launch_jd"),
        F.trim("Launch_Date").alias("launch_date_raw"),
        clean_col("LV_Type").alias("lv_type"),
        clean_col("Variant").alias("variant"),
        clean_col("Agency").alias("agency_raw"),
        clean_col("Launch_Site").alias("launch_site"),
        clean_col("Platform").alias("platform"),
        clean_col("Group").alias("launch_service"),
        clean_col("Category").alias("launch_profile"),
        F.trim("LaunchCode").alias("launch_code"),
        "_snapshot_date",
    )

    # 2. split the launch code
    .withColumn("launch_class",   F.substring("launch_code", 1, 1))
    .withColumn("outcome_letter", F.substring("launch_code", 2, 1))
    .withColumn("pct_raw",
        F.when(F.regexp_extract("launch_code", r"^..(\d+)", 1) != "",
               F.regexp_extract("launch_code", r"^..(\d+)", 1).cast("int")))

    # 3. keep orbital + deep space, real outcomes only
    .filter(F.col("launch_class").isin("O", "D") &
            F.col("outcome_letter").isin("S", "F"))

    # 4. timestamp and labels
    .withColumn("launch_ts",
        F.timestamp_seconds(((F.col("launch_jd") - 2440587.5) * 86400).cast("long")))
    .withColumn("launch_year", F.year("launch_ts"))
    .withColumn("date_uncertain", F.col("launch_date_raw").contains("?"))
    .withColumn("success_pct",
        F.when(F.col("pct_raw").isNotNull(), F.col("pct_raw"))
         .when(F.col("outcome_letter") == "S", 100)
         .otherwise(0))
    .withColumn("failed", F.col("outcome_letter") == "F")
    .withColumn("failed_strict",
        ~((F.col("outcome_letter") == "S") & F.col("pct_raw").isNull()))
    .drop("pct_raw")

    # 5. fixes: uncertain variants and joint agencies
    .withColumn("variant_uncertain", F.col("variant").contains("?"))
    .withColumn("variant",
        F.when(F.regexp_replace("variant", r"\?", "") == "", None)
         .otherwise(F.regexp_replace("variant", r"\?", "")))
    .withColumn("agency_joint",     F.col("agency_raw").contains("/"))
    .withColumn("agency_uncertain", F.col("agency_raw").contains("?"))
    .withColumn("agency",
        F.regexp_replace(F.split("agency_raw", "/").getItem(0), r"\?", ""))
)

# 6. save
(launch.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("launch_reliability.silver.launch"))

print(f"silver.launch: {launch.count():,} rows")
print(launch.columns)

In [0]:
def to_num(col):
    return F.regexp_replace(F.col(col), r"\?", "").cast("double")

orgs_info = (spark.table("launch_reliability.silver.orgs")
    .select(
        F.col("code").alias("agency"),
        F.col("statecode").alias("agency_country"),
        F.when(F.col("class") == "A", "academic")
         .when(F.col("class") == "B", "commercial")
         .when(F.col("class") == "C", "civil_government")
         .when(F.col("class") == "D", "military")
         .alias("provider_class")))

lv_info = (spark.table("launch_reliability.silver.lv")
    .select(
        F.col("lv_name").alias("_lv_name"),
        F.col("lv_variant").alias("_lv_variant"),
        "lv_family",
        "lv_manufacturer",
        F.col("lv_min_stage").cast("int").alias("_min_stage"),
        F.col("lv_max_stage").cast("int").alias("n_core_stages"),
        to_num("length").alias("length_m"),
        to_num("diameter").alias("diameter_m"),
        to_num("launch_mass").alias("launch_mass_t"),
        to_num("to_thrust").alias("liftoff_thrust_kn"),
        to_num("leo_capacity").alias("leo_capacity_kg")))

launch_joined = (spark.table("launch_reliability.silver.launch")
    .join(orgs_info, on="agency", how="left")
    .join(lv_info,
          (F.col("lv_type") == F.col("_lv_name")) &
          F.col("variant").eqNullSafe(F.col("_lv_variant")),
          "left")
    .withColumn("has_strapons", F.col("_min_stage") <= 0)
    .withColumn("is_government",
        F.col("provider_class").isin("civil_government", "military"))
    .drop("_lv_name", "_lv_variant", "_min_stage"))

(launch_joined.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("launch_reliability.silver.launch_joined"))

print(f"silver.launch_joined: {launch_joined.count():,} rows")

Summary: Cleaned the raw GCAT tables and joins them into one launch-level table.

- Launch table: keeps orbital (O) and deep space (D) launches with a known outcome, dropping pad explosions and aborts (7,407 launches). Parses LaunchCode into a success percentage and two labels: failed (follows GCAT’s success/failure letter) and failed_strict (any partial outcome counts as a failure). Builds a timestamp from the Julian Date. Strips ? uncertainty markers from variants and agencies, and uses the first agency listed on joint launches.
- Supporting tables: orgs, lv, lvs, stages, and engines are trimmed, have - converted to null, and get lowercase column names.
Joined table (launch_joined): adds provider class and country from orgs, and rocket family, stage count, strap-on flag, and size, mass, and thrust specs from lv.
- Leakage notes: the O/D category and the launch designation both encode the outcome, so neither is used as a feature.